# Perfect Lap Composite - Interactive Analysis

This notebook walks through the full telemetry pipeline:
1. Load and clean telemetry data
2. Find best mini-sector times
3. Construct theoretical perfect lap
4. Compare with actual qualifying lap
5. Visualize insights

In [1]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

from preprocess       import preprocess
from segment_analysis import analyse
from perfect_lap      import run as build_perfect_lap
from comparison       import run as run_comparison
from visualization    import generate_all

print('Imports OK')

Imports OK


## 1. Generate and Load Data

In [2]:
# Generate sample data if not present
data_path = os.path.join('..', 'data', 'telemetry.csv')
if not os.path.exists(data_path):
    import subprocess
    subprocess.run([sys.executable, os.path.join('..', 'data', 'generate_data.py')], cwd='..', check=True)

df_raw, df_seg = preprocess(data_path)
df_raw.head()

[preprocess] Loading telemetry...
[preprocess] Done. 1482 telemetry rows | 19 laps | 18 mini-sectors.


,lap_id,lap_type,lap_num,mini_sector,timestamp,distance,speed,throttle,brake,gear,gps_x,gps_y,mini_sector_time,timestamp_norm
0,1,fp1,1,1,0.0000,0.0,263.47,0.733,0.245,3,0.00,0.00,3.3662,0.0000
1,1,fp1,1,1,0.8415,65.0,287.15,0.907,0.004,3,0.00,0.00,3.3662,0.8415
2,1,fp1,1,1,1.6831,130.0,291.57,0.943,0.054,3,0.00,0.00,3.3662,1.6831
3,1,fp1,1,1,2.5246,195.0,284.19,0.892,0.105,4,0.00,0.00,3.3662,2.5246
4,1,fp1,1,2,3.3662,260.0,290.62,0.753,0.225,4,246.39,83.02,3.3714,3.3662


## 2. Segment Analysis

In [3]:
best_segments, sector_summ = analyse(df_seg)
best_segments

[segment_analysis] Finding best mini-sector times...
[segment_analysis] Done.

Best segments (first 6):
 mini_sector sector  best_time source_lap_type  source_lap_num
           1     S1     3.2638             fp1               3
           2     S1     3.3514             fp1               3
           3     S1     3.4569             fp1               5
           4     S1     3.4131             fp1               7
           5     S1     3.3827             fp2               2
           6     S1     3.8158             fp2               4

Sector summary:
sector  perfect_time  qualifying_time  delta  delta_pct
    S1       20.6837          21.0039 0.3202      1.548
    S2       20.4979          21.0246 0.5267      2.570
    S3       19.9741          20.4174 0.4433      2.219


,source_lap_id,source_lap_type,source_lap_num,mini_sector,best_time,sector
0,3,fp1,3,1,3.2638,S1
1,3,fp1,3,2,3.3514,S1
2,5,fp1,5,3,3.4569,S1
3,7,fp1,7,4,3.4131,S1
4,10,fp2,2,5,3.3827,S1
5,12,fp2,4,6,3.8158,S1
6,2,fp1,2,7,3.4156,S2
7,14,fp2,6,8,3.5136,S2
8,6,fp1,6,9,3.2358,S2
9,4,fp1,4,10,3.0435,S2


In [4]:
print('Which sessions contributed the most best segments:')
best_segments['source_lap_type'].value_counts()

Which sessions contributed the most best segments:


source_lap_type
fp1           11
fp2            5
qualifying     2
Name: count, dtype: int64

## 3. Perfect Lap

In [5]:
perfect_lap = build_perfect_lap(df_seg, best_segments)
print(perfect_lap.summary())

[perfect_lap] Building perfect lap...
         PERFECT LAP COMPOSITE RESULT
  Perfect Lap Time  : 1:01.156
  Qualifying Lap    : 1:02.446
  Time Saved        : +1.290s
[perfect_lap] Results saved → output\perfect_lap_results.csv
         PERFECT LAP COMPOSITE RESULT
  Perfect Lap Time  : 1:01.156
  Qualifying Lap    : 1:02.446
  Time Saved        : +1.290s


## 4. Comparison

In [6]:
results = run_comparison(perfect_lap, df_seg)
results['mini_sector_comparison']

[comparison] Running lap comparisons...

Mini-sector comparison (first 6 rows):
 mini_sector sector  best_time  qualifying_time  delta
           1     S1     3.2638           3.4080 0.1442
           2     S1     3.3514           3.3894 0.0380
           3     S1     3.4569           3.4730 0.0161
           4     S1     3.4131           3.4454 0.0323
           5     S1     3.3827           3.4437 0.0610
           6     S1     3.8158           3.8444 0.0286

Standard sector comparison:
sector  perfect_time  qualifying_time  delta  delta_pct
    S1       20.6837          21.0039 0.3202      1.548
    S2       20.4979          21.0246 0.5267      2.570
    S3       19.9741          20.4174 0.4433      2.219

Session / driver comparison:
driver_or_session  theoretical_best  actual_best_lap  potential_gain
              fp1           61.3028          62.0862          0.7834
              fp2           61.3274          62.1222          0.7948
       qualifying           61.7777          

,mini_sector,sector,best_time,source_lap_type,source_lap_num,qualifying_time,delta,delta_pct
0,1,S1,3.2638,fp1,3,3.4080,0.1442,4.418
1,2,S1,3.3514,fp1,3,3.3894,0.0380,1.134
2,3,S1,3.4569,fp1,5,3.4730,0.0161,0.466
3,4,S1,3.4131,fp1,7,3.4454,0.0323,0.946
4,5,S1,3.3827,fp2,2,3.4437,0.0610,1.803
5,6,S1,3.8158,fp2,4,3.8444,0.0286,0.750
6,7,S2,3.4156,fp1,2,3.4664,0.0508,1.487
7,8,S2,3.5136,fp2,6,3.6488,0.1352,3.848
8,9,S2,3.2358,fp1,6,3.3147,0.0789,2.438
9,10,S2,3.0435,fp1,4,3.1311,0.0876,2.878


In [7]:
print('Sector summary:')
results['sector_comparison']

Sector summary:


,sector,perfect_time,qualifying_time,delta,delta_pct
0,S1,20.6837,21.0039,0.3202,1.548
1,S2,20.4979,21.0246,0.5267,2.570
2,S3,19.9741,20.4174,0.4433,2.219


## 5. Visualizations

In [8]:
generate_all(
    df_raw,
    results['mini_sector_comparison'],
    results['sector_comparison'],
    results['driver_comparison'],
)
print('Charts saved to ../output/graphs/')

[visualization] Generating charts...
  [viz] Saved → output\graphs\sector_bar_chart.png
  [viz] Saved → output\graphs\mini_sector_bar_chart.png
  [viz] Saved → output\graphs\speed_vs_distance.png
  [viz] Saved → output\graphs\time_delta.png
  [viz] Saved → output\graphs\time_loss_heatmap.png
  [viz] Saved → output\graphs\driver_comparison.png
[visualization] All charts saved to output\graphs/
Charts saved to ../output/graphs/


## 6. Quick Inline Plots
Reproduced here for immediate viewing in the notebook.

In [11]:
ms = results['mini_sector_comparison']
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Cumulative delta
axes[0].plot(ms['mini_sector'], ms['delta'].cumsum(), 'o-', color='#EF9F27')
axes[0].axhline(0, color='gray', lw=0.8, ls='--')
axes[0].set_title('Cumulative Time Delta')
axes[0].set_xlabel('Mini-Sector')
axes[0].set_ylabel('Delta (s)')
axes[0].grid(True, alpha=0.3)

# Bar comparison
x = np.arange(len(ms))
axes[1].bar(x - 0.2, ms['best_time'],       0.4, color='#22c55e', label='Perfect')
axes[1].bar(x + 0.2, ms['qualifying_time'], 0.4, color='#E24B4A', label='Qualifying')
axes[1].set_xticks(x[::3])
axes[1].set_xticklabels([f'MS{i}' for i in ms['mini_sector'].iloc[::3]])
axes[1].set_title('Mini-Sector Times')
axes[1].set_ylabel('Time (s)')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()